<div style="text-align: center;">
    <h1>Regresión Logística — Clasificación de Enfermedad Cardiovascular</h1>
</div>

## 1. Importar librerías

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, StratifiedKFold,
    learning_curve, validation_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve
)

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

## 2. Cargar datos

Se carga directamente la base de datos ya limpia y preprocesada desde el repositorio GitHub del proyecto.

In [ ]:
url = "https://raw.githubusercontent.com/SantCorrea802/Cardiovascular_Disease_Proyecto_Modelos_II/main/dataset/data_cleaned.csv"
data_clean = pd.read_csv(url)

if "id" in data_clean.columns:
    data_clean = data_clean.drop(columns=["id"])
    print("Columna 'id' eliminada.")

print(f"Shape: {data_clean.shape}")
print(f"Columnas: {data_clean.columns.tolist()}")
data_clean.head()

## 3. Separación Train / Validation / Test

Se usa la misma estrategia del EDA: **80% train — 10% val — 10% test**, con estratificación.

In [ ]:
features = data_clean.drop(columns=["cardiovascular_disease"])
target   = data_clean["cardiovascular_disease"]

X_train, X_temp, y_train, y_temp = train_test_split(
    features, target, test_size=0.20, random_state=RANDOM_STATE, stratify=target
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train:      {X_train.shape[0]} muestras")
print(f"Validation: {X_val.shape[0]} muestras")
print(f"Test:       {X_test.shape[0]} muestras")

## 4. Modelo base (baseline)

Se entrena una Regresión Logística con hiperparámetros por defecto.

> **Nota importante:** a diferencia de Random Forest, la Regresión Logística **requiere escalado** de las features. Variables en escalas muy diferentes (ej. presión arterial en mmHg vs variables binarias 0/1) afectan negativamente la convergencia del modelo. Se usa `StandardScaler` dentro de un `Pipeline` para garantizar que el escalado se aprenda solo en train y se aplique correctamente en val y test.

In [ ]:
pipeline_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

pipeline_baseline.fit(X_train, y_train)

y_val_pred_base = pipeline_baseline.predict(X_val)
y_val_prob_base = pipeline_baseline.predict_proba(X_val)[:, 1]

print("=== Baseline — Validación ===")
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred_base):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred_base):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred_base):.4f}")
print(f"F1-Score:  {f1_score(y_val, y_val_pred_base):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_val, y_val_prob_base):.4f}")

## 5. Búsqueda de hiperparámetros — RandomizedSearchCV

Se exploran las principales variantes de regularización de la Regresión Logística.

### Malla de hiperparámetros

| Hiperparámetro | Descripción | Valores explorados |
|---|---|---|
| `model__C` | Inverso de la regularización. C alto = poca regularización | [0.001, 0.01, 0.1, 1, 10, 100] |
| `model__penalty` | Tipo de regularización | ['l1', 'l2'] |
| `model__solver` | Algoritmo de optimización | ['liblinear', 'saga'] |
| `model__class_weight` | Peso de las clases | [None, 'balanced'] |

> **L1 (Lasso):** puede llevar coeficientes a exactamente 0, haciendo selección automática de features.
> **L2 (Ridge):** reduce los coeficientes pero no los elimina. Es el tipo de regularización por defecto.

In [ ]:
param_dist = {
    "model__C":            [0.001, 0.01, 0.1, 1, 10, 100],
    "model__penalty":      ["l1", "l2"],
    "model__solver":       ["liblinear", "saga"],
    "model__class_weight": [None, "balanced"]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipeline_search = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

lr_search = RandomizedSearchCV(
    estimator=pipeline_search,
    param_distributions=param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)

print("\nMejores hiperparámetros encontrados:")
for param, value in lr_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMejor AUC-ROC (CV): {lr_search.best_score_:.4f}")

## 6. Evaluación del modelo optimizado en Validación

In [ ]:
lr_best = lr_search.best_estimator_

y_val_pred = lr_best.predict(X_val)
y_val_prob = lr_best.predict_proba(X_val)[:, 1]

metrics_val = {
    "Accuracy":  accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall":    recall_score(y_val, y_val_pred),
    "F1-Score":  f1_score(y_val, y_val_pred),
    "AUC-ROC":   roc_auc_score(y_val, y_val_prob)
}

print("=== Modelo Optimizado — Validación ===")
for metric, value in metrics_val.items():
    print(f"{metric:10s}: {value:.4f}")

### Matriz de confusión y curva ROC — Validación

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_val, y_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Sin cardiopatía", "Con cardiopatía"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matriz de Confusión — Validación")

fpr, tpr, _ = roc_curve(y_val, y_val_prob)
auc = roc_auc_score(y_val, y_val_prob)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Tasa de Falsos Positivos")
axes[1].set_ylabel("Tasa de Verdaderos Positivos")
axes[1].set_title("Curva ROC — Validación")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 7. Detección de sobre/subajuste

### 7.1 Curva de aprendizaje

Muestra cómo evolucionan train y validación cruzada al aumentar el tamaño del conjunto de entrenamiento.

- **Train alto, val bajo** → overfitting
- **Ambos bajos** → underfitting
- **Ambos altos y cercanos** → modelo bien ajustado ✅

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    lr_best,
    X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, "o-", color="steelblue", label="Train")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="steelblue")
plt.plot(train_sizes, val_mean, "o-", color="tomato", label="Validación cruzada")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="tomato")
plt.xlabel("Tamaño del conjunto de entrenamiento")
plt.ylabel("AUC-ROC")
plt.title("Curva de Aprendizaje — Regresión Logística")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f"AUC-ROC Train (completo):      {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"AUC-ROC Validación (completo): {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")
print(f"Brecha train-val:              {gap:.4f}")
if gap > 0.05:
    print("⚠️  Posible sobreajuste (brecha > 0.05)")
elif val_mean[-1] < 0.70:
    print("⚠️  Posible subajuste (AUC-ROC val < 0.70)")
else:
    print("✅ Modelo bien ajustado")

### 7.2 Curva de validación — efecto de la regularización `C`

Muestra cómo varía el desempeño al cambiar el parámetro `C`.

- **C muy pequeño** → mucha regularización → modelo demasiado simple → underfitting
- **C muy grande** → poca regularización → modelo complejo → riesgo de overfitting
- **C óptimo** → el punto donde val es máximo y la brecha con train es pequeña

In [ ]:
# Extraer los mejores parámetros excepto C
best_params = lr_search.best_params_.copy()
best_C = best_params.pop("model__C")

C_range = [0.001, 0.01, 0.1, 1, 10, 100]

pipeline_vc = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(
        penalty=best_params.get("model__penalty", "l2"),
        solver=best_params.get("model__solver", "liblinear"),
        class_weight=best_params.get("model__class_weight", None),
        random_state=RANDOM_STATE,
        max_iter=1000
    ))
])

train_scores_vc, val_scores_vc = validation_curve(
    pipeline_vc,
    X_train, y_train,
    param_name="model__C",
    param_range=C_range,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1
)

train_m = train_scores_vc.mean(axis=1)
train_s = train_scores_vc.std(axis=1)
val_m   = val_scores_vc.mean(axis=1)
val_s   = val_scores_vc.std(axis=1)

plt.figure(figsize=(9, 5))
plt.semilogx(C_range, train_m, "o-", color="steelblue", label="Train")
plt.fill_between(C_range, train_m - train_s, train_m + train_s, alpha=0.15, color="steelblue")
plt.semilogx(C_range, val_m, "o-", color="tomato", label="Validación cruzada")
plt.fill_between(C_range, val_m - val_s, val_m + val_s, alpha=0.15, color="tomato")
plt.axvline(x=best_C, color="black", linestyle="--", label=f"C óptimo = {best_C}")
plt.xlabel("C (escala logarítmica)")
plt.ylabel("AUC-ROC")
plt.title("Curva de Validación — Efecto de C (regularización)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"C óptimo encontrado: {best_C}")

## 8. Coeficientes del modelo

A diferencia de Random Forest, la Regresión Logística es un modelo **interpretable**: sus coeficientes indican directamente la dirección e intensidad del efecto de cada variable sobre la probabilidad de cardiopatía.

- **Coeficiente positivo** → esa variable aumenta la probabilidad de cardiopatía
- **Coeficiente negativo** → esa variable la reduce
- **Coeficiente cercano a 0** → variable con poco efecto (L1 puede llevarlo exactamente a 0)

In [ ]:
model_step = lr_best.named_steps["model"]
coefs = pd.Series(
    model_step.coef_[0],
    index=X_train.columns
).sort_values()

colors = ["tomato" if c > 0 else "steelblue" for c in coefs]

plt.figure(figsize=(9, 6))
coefs.plot(kind="barh", color=colors, edgecolor="white")
plt.axvline(x=0, color="black", linewidth=0.8)
plt.xlabel("Coeficiente")
plt.title("Coeficientes — Regresión Logística\n(rojo = aumenta riesgo, azul = reduce riesgo)")
plt.tight_layout()
plt.show()

print(coefs.round(4))

## 9. Diagnóstico y corrección

### 9.1 Problema detectado

Se analiza si el modelo presenta Recall bajo (falsos negativos altos), que es la métrica más crítica en este problema clínico.

In [ ]:
cm_val = confusion_matrix(y_val, y_val_pred)
fn = cm_val[1, 0]
recall_actual    = recall_score(y_val, y_val_pred)
precision_actual = precision_score(y_val, y_val_pred)

print("=== Diagnóstico del modelo ===")
print(f"Recall (Validación):    {recall_actual:.4f}")
print(f"Precision (Validación): {precision_actual:.4f}")
print(f"Falsos Negativos:       {fn}")
print(f"\nEl modelo NO detecta el {(1 - recall_actual)*100:.1f}% de los pacientes enfermos.")

if precision_actual - recall_actual > 0.05:
    print("\n⚠️ El modelo es conservador: Precision > Recall.")
    print("   Se explorarán correcciones para mejorar el Recall.")
else:
    print("\n✅ El balance Precision/Recall es aceptable.")

### 9.2 Análisis del umbral de decisión

In [ ]:
precision_curve, recall_curve, thresholds = precision_recall_curve(y_val, y_val_prob)

f1_scores_curve = 2 * (precision_curve[:-1] * recall_curve[:-1]) / \
                  (precision_curve[:-1] + recall_curve[:-1] + 1e-8)
mejor_umbral = thresholds[np.argmax(f1_scores_curve)]

print(f"Umbral original: 0.5000")
print(f"Umbral óptimo:   {mejor_umbral:.4f}")
print(f"Cambio:          {0.5 - mejor_umbral:+.4f} puntos")

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precision_curve[:-1], color="steelblue", label="Precision")
plt.plot(thresholds, recall_curve[:-1], color="tomato", label="Recall")
plt.plot(thresholds, f1_scores_curve, color="green", label="F1-Score")
plt.axvline(x=mejor_umbral, color="black", linestyle="--",
            label=f"Umbral óptimo = {mejor_umbral:.4f}")
plt.axvline(x=0.5, color="gray", linestyle=":", label="Umbral original = 0.5")
plt.xlabel("Umbral de decisión")
plt.ylabel("Métrica")
plt.title("Precision, Recall y F1 según umbral de decisión")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 9.3 Corrección 1 — Ajuste de umbral de decisión

In [ ]:
y_val_pred_umbral = (y_val_prob >= mejor_umbral).astype(int)

print(f"=== Corrección 1: umbral ajustado a {mejor_umbral:.4f} ===")
print(f"Precision: {precision_score(y_val, y_val_pred_umbral):.4f}  (antes: {precision_actual:.4f})")
print(f"Recall:    {recall_score(y_val, y_val_pred_umbral):.4f}  (antes: {recall_actual:.4f})")
print(f"F1-Score:  {f1_score(y_val, y_val_pred_umbral):.4f}  (antes: {f1_score(y_val, y_val_pred):.4f})")
print(f"AUC-ROC:   {roc_auc_score(y_val, y_val_prob):.4f}  (no cambia, no se reentrenó)")

cm_umbral = confusion_matrix(y_val, y_val_pred_umbral)
print(f"\nFalsos Negativos: {cm_umbral[1,0]}  (antes: {fn})")
print(f"Reducción de FN:  {fn - cm_umbral[1,0]} pacientes enfermos ahora detectados correctamente")

### 9.4 Corrección 2 — class_weight='balanced'

Se asigna mayor peso a la clase positiva durante el entrenamiento.

> **Nota:** dado que el dataset está casi perfectamente balanceado (~50/50), el efecto esperado es pequeño, similar a lo observado en el modelo Random Forest.

In [ ]:
balance = y_train.value_counts(normalize=True)
print("Balance de clases en train:")
print(f"  Sin cardiopatía (0): {balance[0]*100:.1f}%")
print(f"  Con cardiopatía (1): {balance[1]*100:.1f}%")

best_params_bal = lr_search.best_params_.copy()
best_params_bal["model__class_weight"] = "balanced"

pipeline_balanced = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(
        C=best_params_bal["model__C"],
        penalty=best_params_bal["model__penalty"],
        solver=best_params_bal["model__solver"],
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=1000
    ))
])
pipeline_balanced.fit(X_train, y_train)

y_val_pred_bal = pipeline_balanced.predict(X_val)
y_val_prob_bal = pipeline_balanced.predict_proba(X_val)[:, 1]

print(f"\n=== Corrección 2: class_weight='balanced' ===")
print(f"Precision: {precision_score(y_val, y_val_pred_bal):.4f}  (antes: {precision_actual:.4f})")
print(f"Recall:    {recall_score(y_val, y_val_pred_bal):.4f}  (antes: {recall_actual:.4f})")
print(f"F1-Score:  {f1_score(y_val, y_val_pred_bal):.4f}  (antes: {f1_score(y_val, y_val_pred):.4f})")
print(f"AUC-ROC:   {roc_auc_score(y_val, y_val_prob_bal):.4f}")

### 9.5 Tabla comparativa y conclusión

In [ ]:
comparativa = pd.DataFrame({
    "Original (umbral=0.5)": [
        accuracy_score(y_val, y_val_pred),
        precision_score(y_val, y_val_pred),
        recall_score(y_val, y_val_pred),
        f1_score(y_val, y_val_pred),
        roc_auc_score(y_val, y_val_prob)
    ],
    f"Umbral ajustado ({mejor_umbral:.2f})": [
        accuracy_score(y_val, y_val_pred_umbral),
        precision_score(y_val, y_val_pred_umbral),
        recall_score(y_val, y_val_pred_umbral),
        f1_score(y_val, y_val_pred_umbral),
        roc_auc_score(y_val, y_val_prob)
    ],
    "class_weight=balanced": [
        accuracy_score(y_val, y_val_pred_bal),
        precision_score(y_val, y_val_pred_bal),
        recall_score(y_val, y_val_pred_bal),
        f1_score(y_val, y_val_pred_bal),
        roc_auc_score(y_val, y_val_prob_bal)
    ]
}, index=["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"])

print("=== Comparativa de correcciones — Validación ===")
print(comparativa.round(4))

recall_umbral = recall_score(y_val, y_val_pred_umbral)
recall_bal    = recall_score(y_val, y_val_pred_bal)
mejor_correccion = "Umbral ajustado" if recall_umbral >= recall_bal else "class_weight=balanced"

print(f"\n=== Conclusión ===")
print(f"La corrección más efectiva fue: {mejor_correccion}")
print(f"Recall mejoró de {recall_actual:.4f} → {max(recall_umbral, recall_bal):.4f}")

## 10. Evaluación final en Test

Se evalúa el modelo optimizado sobre el conjunto de test.

In [ ]:
y_test_pred  = lr_best.predict(X_test)
y_test_prob  = lr_best.predict_proba(X_test)[:, 1]
y_train_pred = lr_best.predict(X_train)
y_train_prob = lr_best.predict_proba(X_train)[:, 1]

metrics_test = {
    "Accuracy":  accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall":    recall_score(y_test, y_test_pred),
    "F1-Score":  f1_score(y_test, y_test_pred),
    "AUC-ROC":   roc_auc_score(y_test, y_test_prob)
}

comparison = pd.DataFrame({
    "Train": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred),
        roc_auc_score(y_train, y_train_prob)
    ],
    "Validación": list(metrics_val.values()),
    "Test":       list(metrics_test.values())
}, index=["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"])

print("=== Comparativa Train / Validación / Test ===")
print(comparison.round(4))

### Matriz de confusión y curva ROC — Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_test = confusion_matrix(y_test, y_test_pred)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=["Sin cardiopatía", "Con cardiopatía"])
disp_test.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matriz de Confusión — Test")

fpr_t, tpr_t, _ = roc_curve(y_test, y_test_prob)
auc_t = roc_auc_score(y_test, y_test_prob)
axes[1].plot(fpr_t, tpr_t, color="steelblue", lw=2, label=f"AUC = {auc_t:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Tasa de Falsos Positivos")
axes[1].set_ylabel("Tasa de Verdaderos Positivos")
axes[1].set_title("Curva ROC — Test")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 11. Exportar modelo entrenado

In [ ]:
import joblib

joblib.dump(lr_best, "exported_logistic_regression.joblib")
print("Modelo exportado como 'exported_logistic_regression.joblib'")